In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Download NLTK stopwords (run once)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# ========================== 1. LOAD DATA ==========================
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

# Add labels
fake['label'] = 1   # 1 = Fake
true['label'] = 0   # 0 = Real

# Combine & shuffle
df = pd.concat([fake, true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset loaded!")
print(df['label'].value_counts())

# ========================== 2. TEXT PREPROCESSING ==========================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)          # remove punctuation & numbers
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

print("Cleaning text... (this takes ~1-2 minutes)")
df['clean_text'] = df['text'].apply(clean_text)

# ========================== 3. FEATURE ENGINEERING & SPLIT ==========================
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorizer (turns text into numbers)
vectorizer = TfidfVectorizer(max_features=5000)   # limit to 5000 most important words
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print("Vectorization done!")

# ========================== 4. TRAIN MODEL ==========================
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train_vec, y_train)

# ========================== 5. EVALUATE ==========================
y_pred = model.predict(X_test_vec)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Fake News Detector')
plt.show()

# ========================== 6. SAVE MODEL & VECTORIZER ==========================
joblib.dump(model, 'fake_news_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Model saved successfully! Ready to use.")